# PGA-UNet — Test & Visualization
**Checkpoint**: `1Mv-rUPI7KGmYemd27hmKbJQRHc4ZKB9z`

Chạy tuần tự: **Setup → Test → Visualization**

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────
%cd /content
import os, gdown

if not os.path.exists('Prompt-Guided-XRay-Segmentation'):
    !git clone -b TN_B_ON https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git
else:
    !cd Prompt-Guided-XRay-Segmentation && git pull -q

!gdown 1fU7KPln7joaa3EZZtGn-VKeg9i4AmPG3 -O /content/dataset_BTXRD.zip -q
!unzip -oq /content/dataset_BTXRD.zip -d /content/
!rsync -a /content/dataset_BTXRD/ /content/Prompt-Guided-XRay-Segmentation/dataset_BTXRD/ 2>/dev/null

%cd /content/Prompt-Guided-XRay-Segmentation
!pip install -q tqdm opencv-python matplotlib gdown scipy

CKPT_ID   = '1Mv-rUPI7KGmYemd27hmKbJQRHc4ZKB9z'
CKPT_PATH = 'checkpoints/pga_unet_expB_best.pth'
os.makedirs('checkpoints', exist_ok=True)
if not os.path.exists(CKPT_PATH):
    gdown.download(f'https://drive.google.com/uc?id={CKPT_ID}', CKPT_PATH, quiet=False)
assert os.path.exists(CKPT_PATH)
print(f'\n✅ Setup xong  |  {os.path.getsize(CKPT_PATH)//1024} KB')

In [ ]:
# ── Test: 3 prompt modes ─────────────────────────────────────────
import os, cv2, csv, glob
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt

from dataset import BTXRD_Dataset
from models.networks.prompt_unet_2D import PGA_UNet

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 512
IMG_DIR  = 'dataset_BTXRD/test/images'
JSON_DIR = 'dataset_BTXRD/test/annotations'
os.makedirs('results', exist_ok=True)

model = PGA_UNet(in_channels=1, n_classes=1, use_encoder_prompt=True).to(DEVICE)
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE, weights_only=True))
model.eval()
print(f'✅ Model loaded  device={DEVICE}')

def extract_lcc(m):
    if m.sum()==0: return m
    n,lbl,st,_=cv2.connectedComponentsWithStats(m.astype(np.uint8),8)
    return m if n<=1 else (lbl==(1+np.argmax(st[1:,cv2.CC_STAT_AREA]))).astype(np.float32)

def calc_hd95(pred, gt):
    p,g=pred.astype(bool),gt.astype(bool)
    if not p.any() and not g.any(): return 0.0
    if not p.any() or not g.any(): return float(IMG_SIZE)
    pe=p^binary_erosion(p); ge=g^binary_erosion(g)
    d1=distance_transform_edt(~ge)[pe]; d2=distance_transform_edt(~pe)[ge]
    return float(IMG_SIZE) if not len(d1) or not len(d2) else float(max(np.percentile(d1,95),np.percentile(d2,95)))

def calc_metrics(prob_np, gt_np):
    pm=extract_lcc((prob_np>0.5).astype(np.float32))
    gm=(gt_np>0.5).astype(np.float32); eps=1e-6
    tp=(pm*gm).sum(); fp=(pm*(1-gm)).sum(); fn=((1-pm)*gm).sum()
    hd95=calc_hd95(pm,gm)
    if gm.sum()==0 or pm.sum()==0: cbl=0.0
    else:
        ys,xs=np.where(gm>0.5); yp,xp=np.where(pm>0.5)
        cbl=float(np.clip(1.-np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/(np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+eps),0,1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)),iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=float((tp+eps)/(tp+fp+eps)),recall=float((tp+eps)/(tp+fn+eps)),
                hd95=hd95,cbl=cbl,mask=pm)

KEYS  = ['dice','iou','precision','recall','hd95','cbl']
HDRS  = ['Dice↑','IoU↑','Prec↑','Rec↑','HD95↓','CBL↑']
MODES = ['zoom_out','shift','mixed_7_3']
all_records = {}
csv_rows    = []

for mode in MODES:
    ds     = BTXRD_Dataset(IMG_DIR, JSON_DIR, img_size=IMG_SIZE, is_train=False, prompt_mode=mode)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
    recs   = []
    for img_t, mask_t, prompt_t in tqdm(loader, desc=f'[{mode}]'):
        gt_np = mask_t[0,0].numpy()
        with torch.no_grad():
            prob = torch.sigmoid(model(img_t.to(DEVICE), prompt_t.to(DEVICE)))[0,0].cpu().numpy()
        m = calc_metrics(prob, gt_np)
        recs.append(dict(img=img_t[0,0].numpy(), gt=gt_np, prob=prob,
                         prompt=prompt_t[0,0].numpy(), **{k:m[k] for k in KEYS}))
    all_records[mode] = recs
    m_avg = {k: np.mean([r[k] for r in recs]) for k in KEYS}
    csv_rows.append([mode]+[f'{m_avg[k]:.4f}' for k in KEYS]+[str(len(recs))])

# Bảng kết quả
bar='='*74
print(f'\n{bar}\n  PGA-UNet  |  N=248\n{bar}')
print(f"  {'Mode':<16}"+''.join(f'{h:>8}' for h in HDRS))
print(f"  {'-'*70}")
for row in csv_rows:
    print(f"  {row[0]:<16}"+''.join(f'{row[i+1]:>8}' for i in range(len(KEYS))))
print(bar)

with open('results/pga_extended_test_results.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['mode']+KEYS+['N']); w.writerows(csv_rows)
print('\n✅ CSV: results/pga_extended_test_results.csv')

In [ ]:
# ── Visualization: 3 ảnh × 3 mode ─────────────────────────────────────────────
import matplotlib.pyplot as plt, json as _json
from matplotlib.patches import Rectangle

assert 'model' in dir() and 'calc_metrics' in dir(), '❌ Chạy cell Test trước'

VIS_IMAGES  = ['IMG001768', 'IMG001538', 'IMG001100']
MODE_LABELS = {'zoom_out':'Zoom-Out (~30%)', 'shift':'Shift (~30%)', 'mixed_7_3':'Mixed 70/30'}
MODE_COLORS = {'zoom_out':'limegreen', 'shift':'tomato', 'mixed_7_3':'gold'}

def _zoom_bbox(pts, sx, sy, r=0.30, S=512):
    xs, ys = pts[:,0]*sx, pts[:,1]*sy
    x1,x2,y1,y2 = xs.min(),xs.max(),ys.min(),ys.max()
    bw,bh = x2-x1, y2-y1
    return [max(0,x1-bw*r), max(0,y1-bh*r), min(S,x2+bw*r), min(S,y2+bh*r)]

def _shift_bbox(pts, sx, sy, r=0.30, shift=0.30, S=512):
    xs, ys = pts[:,0]*sx, pts[:,1]*sy
    x1,x2,y1,y2 = xs.min(),xs.max(),ys.min(),ys.max()
    bw,bh = x2-x1, y2-y1
    bx1=max(0,x1-bw*r); bx2=min(S,x2+bw*r)
    by1=max(0,y1-bh*r); by2=min(S,y2+bh*r)
    return [max(0,bx1+bw*shift*0.8), max(0,by1+bh*shift*0.3),
            min(S,bx2+bw*shift*0.8), min(S,by2+bh*shift*0.3)]

def _gauss_heatmap(bbox, S=512):
    """Giống create_plateau_heatmap của dataset: kernel cố định (31,31), không normalize."""
    x1 = max(0, int(bbox[0]) - 5)
    y1 = max(0, int(bbox[1]) - 5)
    x2 = min(S, int(bbox[2]) + 5)
    y2 = min(S, int(bbox[3]) + 5)
    hm = np.zeros((S, S), dtype=np.float32)
    if x2 > x1 and y2 > y1:
        hm[y1:y2, x1:x2] = 1.0
        hm = cv2.GaussianBlur(hm, (31, 31), 0)
    return hm

def _overlay(rgb_bg, mask, color_rgb, alpha=0.55):
    out = rgb_bg.copy().astype(np.float32)
    out[mask > 0.5] = out[mask > 0.5]*(1-alpha) + np.array(color_rgb, dtype=np.float32)*alpha
    return np.clip(out, 0, 255).astype(np.uint8)

def _metric_label(ax, text):
    ax.text(0.5, 0.97, text, transform=ax.transAxes,
            ha='center', va='top', fontsize=8, fontweight='bold', color='white',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.55))

S = IMG_SIZE
COL_TITLES = ['Prompt BBox\n(bác sĩ vẽ)', 'Gaussian Heatmap\n(model nhận vào)',
              'Prediction (đỏ)', 'Ground Truth (xanh)', 'Overlap']

for vis_base in VIS_IMAGES:
    img_path  = next((f'{IMG_DIR}/{vis_base}{e}' for e in ['.png','.jpg','.jpeg']
                      if os.path.exists(f'{IMG_DIR}/{vis_base}{e}')), None)
    json_path = f'{JSON_DIR}/{vis_base}.json'
    if not img_path or not os.path.exists(json_path):
        print(f'⚠️  Bỏ qua {vis_base}'); continue

    img_bgr = cv2.imread(img_path)
    H0, W0  = img_bgr.shape[:2]
    img_rs  = cv2.resize(img_bgr, (S, S))
    sx, sy  = S/W0, S/H0

    img_gray = cv2.cvtColor(img_rs, cv2.COLOR_BGR2GRAY)
    img_rgb  = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB)
    img_norm = (img_gray.astype(np.float32)/255. - 0.5) / 0.5

    data     = _json.load(open(json_path))
    polygons = [s for s in data.get('shapes',[]) if s.get('shape_type')=='polygon']
    if not polygons: continue
    pts = np.array(polygons[0]['points'], dtype=np.float32)

    pts_rs = pts.copy(); pts_rs[:,0] *= sx; pts_rs[:,1] *= sy
    gt = np.zeros((S,S), dtype=np.uint8)
    cv2.fillPoly(gt, [pts_rs.astype(np.int32).reshape(-1,1,2)], 255)
    gt_f = (gt/255.).astype(np.float32)

    bz = _zoom_bbox(pts, sx, sy, r=0.30, S=S)
    bs = _shift_bbox(pts, sx, sy, r=0.30, shift=0.30, S=S)
    bboxes   = {'zoom_out': bz, 'shift': bs, 'mixed_7_3': bz}
    heatmaps = {m: _gauss_heatmap(bboxes[m], S) for m in MODES}

    img_t = torch.from_numpy(img_norm).unsqueeze(0).unsqueeze(0).float().to(DEVICE)
    results = {}
    for mode in MODES:
        pm_t = torch.from_numpy(heatmaps[mode]).unsqueeze(0).unsqueeze(0).float().to(DEVICE)
        with torch.no_grad():
            prob = torch.sigmoid(model(img_t, pm_t))[0,0].cpu().numpy()
        m = calc_metrics(prob, gt_f)
        results[mode] = {**m, 'bbox': bboxes[mode], 'heatmap': heatmaps[mode]}

    # ── Plot 3 hàng × 5 cột ─────────────────────────────────────────
    fig, axes = plt.subplots(3, 5, figsize=(22, 12))
    fig.suptitle(f'PGA-UNet — {vis_base}', fontsize=13, fontweight='bold', y=1.01)
    for j, t in enumerate(COL_TITLES):
        axes[0, j].set_title(t, fontsize=9, fontweight='bold')

    for row, mode in enumerate(MODES):
        r     = results[mode]
        color = MODE_COLORS[mode]
        bbox  = r['bbox']

        # Col 0: ảnh gốc + bbox rectangle
        ax = axes[row, 0]
        ax.imshow(img_gray, cmap='gray', vmin=0, vmax=255)
        x1,y1,x2,y2 = bbox
        rect = Rectangle((x1,y1), x2-x1, y2-y1,
                          linewidth=2.5, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.set_ylabel(MODE_LABELS[mode], fontsize=10, fontweight='bold',
                      color=color, rotation=0, labelpad=85, va='center')
        ax.axis('off')

        # Col 1: heatmap overlay lên ảnh gốc (đúng style training)
        ax = axes[row, 1]
        ax.imshow(img_gray, cmap='gray', vmin=0, vmax=255)
        hm = r['heatmap']
        ax.imshow(np.ma.masked_where(hm < 0.05, hm), cmap='hot', vmin=0, vmax=hm.max(), alpha=0.6)
        ax.axis('off')

        # Col 2: prediction overlay đỏ + Dice / IoU / CBL
        ax = axes[row, 2]
        ax.imshow(_overlay(img_rgb, r['mask'], [220, 60, 60]))
        _metric_label(ax, f"Dice={r['dice']:.3f}  IoU={r['iou']:.3f}  CBL={r['cbl']:.3f}")
        ax.axis('off')

        # Col 3: GT overlay xanh
        ax = axes[row, 3]
        ax.imshow(_overlay(img_rgb, gt_f, [30, 120, 255]))
        ax.axis('off')

        # Col 4: overlap diff + Precision / Recall / HD95
        diff = img_rgb.copy()
        pb, gb = r['mask'] > 0.5, gt_f > 0.5
        diff[gb & ~pb] = [30,  180,  30]
        diff[pb & ~gb] = [220,  60,  60]
        diff[pb &  gb] = [220, 200,   0]
        ax = axes[row, 4]
        ax.imshow(diff)
        _metric_label(ax, f"Prec={r['precision']:.3f}  Rec={r['recall']:.3f}  HD95={r['hd95']:.1f}px")
        ax.axis('off')

    plt.tight_layout()
    out = f'vis_pga_{vis_base}.png'
    plt.savefig(out, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'✅ {out}')
